# PND Thesis — Fine-tuning on Kaggle GPU
Runs: pretrain verification + fine-tune at_pre, at_scratch, cnnbilstm × 3 seeds + label-efficiency
Expected runtime: ~2-3 hours on T4 GPU

In [ ]:
# ── Step 1: Clone latest code from GitHub ──────────────────────────────────
!git clone https://github.com/thinhpd1906/detecting_pump_dump_crypto_real_time.git thesis
%cd thesis
!git log --oneline -3

In [ ]:
# ── Step 2: Install dependencies ───────────────────────────────────────────
!pip install -r requirements.txt -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Step 3: Copy processed data from Kaggle dataset ────────────────────────
import os, shutil, zipfile

# Data comes from Kaggle dataset input
data_src = '/kaggle/input/pnd-thesis-processed'
os.makedirs('data/processed', exist_ok=True)

# Find and extract the zip or copy directly
if os.path.exists(f'{data_src}/pnd-processed.zip'):
    print('Extracting zip...')
    with zipfile.ZipFile(f'{data_src}/pnd-processed.zip', 'r') as z:
        z.extractall('.')
    print('Extracted')
else:
    # Files extracted directly by Kaggle
    shutil.copytree(data_src, 'data/processed', dirs_exist_ok=True)
    print('Copied from dataset')

# Verify
for mode in ['none', 'static', 'adaptive']:
    path = f'data/processed/{mode}/train.npz'
    if os.path.exists(path):
        import numpy as np
        d = np.load(path)
        print(f'{mode}: train={len(d["y"]):,} windows, pos={d["y"].sum():,}')
    else:
        print(f'WARNING: {path} not found')

In [ ]:
# ── Step 4: Copy pretrained checkpoint ─────────────────────────────────────
ckpt_src = '/kaggle/input/pnd-thesis-checkpoint/at_pretrained.pt'
os.makedirs('artifacts/adaptive', exist_ok=True)

if os.path.exists(ckpt_src):
    shutil.copy(ckpt_src, 'artifacts/adaptive/at_pretrained.pt')
    size = os.path.getsize('artifacts/adaptive/at_pretrained.pt') / 1e6
    print(f'Checkpoint loaded: {size:.1f} MB')
else:
    print('WARNING: checkpoint not found — will run pretrain from scratch')
    print('This adds ~25 minutes to runtime')

In [ ]:
# ── Step 5: Run pretrain if no checkpoint ──────────────────────────────────
if not os.path.exists('artifacts/adaptive/at_pretrained.pt'):
    print('Running pretrain from scratch (5 epochs on GPU)...')
    !python -u -m src.training.pretrain --clean adaptive --epochs 5
else:
    print('Checkpoint exists — skipping pretrain')
    print('Verifying checkpoint...')
    ckpt = torch.load('artifacts/adaptive/at_pretrained.pt', map_location='cpu')
    print(f'  Epoch saved: {ckpt.get("epoch", "unknown")}')
    print(f'  Model config: {ckpt.get("model_cfg", {})}')

In [ ]:
# ── Step 6: Fine-tune AT pretrained (OURS) — 3 seeds ──────────────────────
print('='*60)
print('FINE-TUNING: AT pretrained (OURS)')
print('='*60)
for seed in [0, 1, 2]:
    print(f'\n--- seed {seed} ---')
    !python -u -m src.training.finetune \
        --clean adaptive \
        --arch at \
        --pretrained \
        --seed {seed}
print('\nat_pre done')

In [ ]:
# ── Step 7: Fine-tune AT from scratch (TWIN) — 3 seeds ────────────────────
print('='*60)
print('FINE-TUNING: AT from scratch (twin comparison)')
print('='*60)
for seed in [0, 1, 2]:
    print(f'\n--- seed {seed} ---')
    !python -u -m src.training.finetune \
        --clean adaptive \
        --arch at \
        --seed {seed}
print('\nat_scratch done')

In [ ]:
# ── Step 8: Fine-tune CNN-BiLSTM — 3 seeds ────────────────────────────────
print('='*60)
print('FINE-TUNING: CNN-BiLSTM baseline')
print('='*60)
for seed in [0, 1, 2]:
    print(f'\n--- seed {seed} ---')
    !python -u -m src.training.finetune \
        --clean adaptive \
        --arch cnnbilstm \
        --seed {seed}
print('\ncnnbilstm done')

In [ ]:
# ── Step 9: Label-efficiency study ────────────────────────────────────────
print('='*60)
print('LABEL-EFFICIENCY STUDY')
print('='*60)
!python -u -m src.training.label_efficiency --clean adaptive --seeds 3
print('\nLabel-efficiency done')

In [ ]:
# ── Step 10: Aggregate results — Tables 1 and 2 ───────────────────────────
print('='*60)
print('AGGREGATING RESULTS')
print('='*60)
!python -m src.training.evaluate --clean adaptive

In [ ]:
# ── Step 11: Package all results for download ──────────────────────────────
import zipfile, glob

output_zip = '/kaggle/working/pnd-results.zip'
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    # All result JSONs
    for f in glob.glob('artifacts/**/*_test.json', recursive=True):
        zf.write(f)
    # Label efficiency
    for f in glob.glob('artifacts/**/label_efficiency.json', recursive=True):
        zf.write(f)
    # Model checkpoints (at_pre and at_scratch only — CNN too big)
    for f in glob.glob('artifacts/**/at_pre*.pt', recursive=True):
        zf.write(f)
    for f in glob.glob('artifacts/**/at_scratch*.pt', recursive=True):
        zf.write(f)
    # Updated pretrained checkpoint
    if os.path.exists('artifacts/adaptive/at_pretrained.pt'):
        zf.write('artifacts/adaptive/at_pretrained.pt')

size = os.path.getsize(output_zip) / 1e6
print(f'Results packaged: {output_zip} ({size:.1f} MB)')
print('\nFiles included:')
with zipfile.ZipFile(output_zip, 'r') as zf:
    for name in zf.namelist():
        print(f'  {name}')

In [ ]:
# ── Step 12: Gate 3 quick verdict ─────────────────────────────────────────
import json, glob

pre_files = sorted(glob.glob('artifacts/adaptive/at_pre_s*_test.json'))
scr_files = sorted(glob.glob('artifacts/adaptive/at_scratch_s*_test.json'))

if pre_files and scr_files:
    pre_f1s = [json.load(open(f))['f1'] for f in pre_files]
    scr_f1s = [json.load(open(f))['f1'] for f in scr_files]
    import numpy as np
    pre_mean, pre_std = np.mean(pre_f1s), np.std(pre_f1s)
    scr_mean, scr_std = np.mean(scr_f1s), np.std(scr_f1s)
    print('='*60)
    print('GATE 3 VERDICT')
    print(f'  at_pre    F1 = {pre_mean:.4f} ± {pre_std:.4f}')
    print(f'  at_scratch F1 = {scr_mean:.4f} ± {scr_std:.4f}')
    if pre_mean > scr_mean + max(pre_std, scr_std):
        print('  GATE 3: PASS — pretraining helps (gap outside std)')
    elif pre_mean > scr_mean:
        print('  GATE 3: MARGINAL — check label-efficiency at 10% labels')
    else:
        print('  GATE 3: FAIL — pretraining did not help')
    print('='*60)
else:
    print('No result files found — check fine-tuning steps above')